## *Clone* github


In [1]:
  !python3 -c "from osgeo import gdal; print(f'GDAL {gdal.__version__} OK')"


GDAL 3.10.3 OK


!git clone https://github.com/ultralytics/ultralytics.git

## Install libraries

%pip install ultralytics supervision "ultralytics<=8.3.40"


In [2]:
# Fix: stub out _lzma module (not needed for YOLO training)
import sys
import types

class _LZMAError(Exception):
    pass

class _LZMACompressor:
    def __init__(self, *args, **kwargs):
        raise _LZMAError("lzma is not available")

class _LZMADecompressor:
    def __init__(self, *args, **kwargs):
        raise _LZMAError("lzma is not available")

mod = types.ModuleType('_lzma')
mod.LZMACompressor = _LZMACompressor
mod.LZMADecompressor = _LZMADecompressor
mod.LZMAError = _LZMAError
mod.FORMAT_AUTO = 0
mod.FORMAT_XZ = 1
mod.FORMAT_ALONE = 2
mod.FORMAT_RAW = 3
mod.CHECK_NONE = 0
mod.CHECK_CRC32 = 1
mod.CHECK_CRC64 = 4
mod.CHECK_SHA256 = 10
mod.CHECK_ID_MAX = 15
mod.CHECK_UNKNOWN = 16
mod.MF_HC3 = 0x03
mod.MF_HC4 = 0x04
mod.MF_BT2 = 0x12
mod.MF_BT3 = 0x13
mod.MF_BT4 = 0x14
mod.MODE_FAST = 1
mod.MODE_NORMAL = 2
mod.PRESET_DEFAULT = 6
mod.PRESET_EXTREME = (1 << 31)
mod._encode_filter_properties = lambda props: b''
mod._decode_filter_properties = lambda fid, data: {}
sys.modules['_lzma'] = mod

# Patch tarfile.is_tarfile to handle missing lzma gracefully
import tarfile
_orig_is_tarfile = tarfile.is_tarfile
def _safe_is_tarfile(name):
    try:
        return _orig_is_tarfile(name)
    except Exception:
        return False
tarfile.is_tarfile = _safe_is_tarfile

print("_lzma stub + tarfile patch installed")

_lzma stub + tarfile patch installed


In [3]:
import ultralytics
ultralytics.checks()


Ultralytics 8.4.14 🚀 Python-3.13.7 torch-2.10.0+cu128 CUDA:0 (NVIDIA L40S, 45458MiB)
Setup complete ✅ (48 CPUs, 60.3 GB RAM, 550.9/785.3 GB disk)


In [4]:
  !apt-get update && apt-get install -y liblzma-dev
  !pip install --force-reinstall --no-binary :all: lzma


Czytanie list pakietów... Gotowe
E: Nie udało się otworzyć pliku blokady /var/lib/apt/lists/lock - open (13: Brak dostępu)
E: Nie udało się zablokować katalogu /var/lib/apt/lists/
W: Problem przy odlinkowywaniu pliku /var/cache/apt/pkgcache.bin - RemoveCaches (13: Brak dostępu)
W: Problem przy odlinkowywaniu pliku /var/cache/apt/srcpkgcache.bin - RemoveCaches (13: Brak dostępu)
ERROR: Could not find a version that satisfies the requirement lzma (from versions: none)
ERROR: No matching distribution found for lzma


## Check GPU

In [5]:

import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

2.10.0+cu128
CUDA available: True
CUDA version: 12.8


In [6]:
!python3 -c "import lzma; print('lzma available')"


lzma available


In [7]:
!dpkg -l | grep liblzma    # Debian/Ubuntu


ii  liblzma-dev:amd64                                5.8.1-1build2                              amd64        XZ-format compression library - development files
ii  liblzma5:amd64                                   5.8.1-1build2                              amd64        XZ-format compression library


*kursywa*## Upload dataset and unzip it

!mkdir -p datasets/qala_1_class/labels/train datasets/qala_1_class/labels/val datasets/qala_1_class/labels/test
!mkdir -p datasets/qala_1_class/images/train datasets/qala_1_class/images/val datasets/qala_1_class/images/test


import zipfile
with zipfile.ZipFile("/content/hidden_in_the_sands_qala_v_1.0.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/datasets")



import zipfile
with zipfile.ZipFile("/content/runs.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/runs")


## Load the pretrained model

In [8]:
from ultralytics import YOLO
#model = YOLO("yolov9t.pt")# load weights
model = YOLO("yolo11n.yaml").load("yolo11n.pt")  # build from YAML and transfer weights


Transferred 499/499 items from pretrained weights


In [9]:
#results = model.train(data="datasets/qala_1_class.yaml", batch=8, epochs=100, imgsz=512)
results = model.train(data="datasets/hidden_in_the_sands_qala_v_1.0.yaml", batch=8, epochs=200, imgsz=512)


Ultralytics 8.4.14 🚀 Python-3.13.7 torch-2.10.0+cu128 CUDA:0 (NVIDIA L40S, 45458MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/hidden_in_the_sands_qala_v_1.0.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train29, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati

In [10]:
import os 

model = YOLO("runs/detect/train25/weights/best.pt")  # load a custom model

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # a list contains map50-95 of each category

Ultralytics 8.4.14 🚀 Python-3.13.7 torch-2.10.0+cu128 CUDA:0 (NVIDIA L40S, 45458MiB)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4960.2±722.0 MB/s, size: 418.1 KB)
val: Scanning /home/nazar/Workspace/qala/datasets/labels/val.cache... 36 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 36/36 15.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.0it/s 1.5s1.1s
                   all         36         36      0.971          1      0.994      0.818
Speed: 4.5ms preprocess, 3.8ms inference, 0.0ms loss, 4.5ms postprocess per image
Results saved to /home/nazar/Workspace/qala/runs/detect/val5


array([    0.81757])

In [11]:
import zipfile
import os

folder_path = "/content/runs"
zip_path = "/content/runs.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, folder_path)
            zipf.write(full_path, arcname)


FileNotFoundError: [Errno 2] No such file or directory: '/content/runs.zip'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the CSV file
data_csv = input("Enter the reference file path: ").strip('"')
df = pd.read_csv(data_csv)  # Ensure the correct path
# Display the first few rows to understand its structure
df.head()




In [ ]:
# Plot loss curves
plt.figure(figsize=(8, 5))
plt.plot(df['epoch'], df['train/cls_loss'], label='Training Loss', marker='o')
plt.plot(df['epoch'], df['val/cls_loss'], label='Validation Loss', marker='o')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Curve")
plt.legend()
plt.grid()



# Save the plot as an image
plt.savefig("runs/detect/train/loss_curve2.png", dpi=300)
plt.show()

!git clone https://github.com/nazarb/qala_utils

pip install segment-geospatial

pip install leafmap

In [13]:
from qala_utils.utils import *

scipy or skimage not available. Binary mask conversion may not work.


To use SamGeo 3, install it as:
	pip install segment-geospatial[samgeo3]


In [14]:
# Then re-run your imports:
import logging
import os
from pathlib import Path
from qala_utils.utils import (
    ImagePreprocessor,
    TileGenerator,
    YOLODetector,
    DetectionPostprocessor,
    QalaPipeline  #
)

In [15]:
## Install GDAL Python bindings
!pip install GDAL==$(gdal-config --version) 2>/dev/null || pip install pygdal==$(gdal-config --version).* 2>/dev/null || pip install gdal
!python3 -c "from osgeo import gdal; print(f'GDAL {gdal.__version__} OK')"

GDAL 3.10.3 OK


In [16]:

import logging
from pathlib import Path
from osgeo import gdal
from qala_utils.utils import (
    ImagePreprocessor,
    TileGenerator,
    YOLODetector,
    DetectionPostprocessor,
    QalaPipeline
)

import numpy as np

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)


In [20]:

# ============================================
# CONFIGURATION
# ============================================
rgb_path = "datasets/D3C1218-401317F012_a_geo.tif"
model_path = "runs/detect/train29/weights/best.pt"
tile_size = 1024
output_dir = "results/"

# Create directories
Path(output_dir).mkdir(exist_ok=True)
Path("temp").mkdir(exist_ok=True)

# ============================================
# STEP 1-4: Preprocess Image
# ============================================
preprocessor = ImagePreprocessor()

# 1. Grayscale
gray_path = preprocessor.convert_to_grayscale_8bit(
    rgb_path,
    "temp/image_gray.tif"
)

# 2. Reproject to EPSG:4326
proj_path = preprocessor.reproject_to_4326(
    gray_path,
    "temp/image_4326.tif"
)

# 3. Clip/pad to regular size (automatic!)
clip_path = preprocessor.clip_or_pad_to_regular_size(
    proj_path,
    "temp/image_clipped.tif",
    tile_size=tile_size,
    use_padding=False  # Clip mode
)

# 4. Extract reference geotransform
ref_ds = gdal.Open(clip_path)
reference_geotransform = ref_ds.GetGeoTransform()
clip_width = ref_ds.RasterXSize
clip_height = ref_ds.RasterYSize
ref_ds = None

print(f"Reference image: {clip_width}×{clip_height}")
print(f"Reference geotransform: {reference_geotransform}")

# ============================================
# STEP 5: Generate Tiles
# ============================================
tile_gen = TileGenerator(tile_size=tile_size, overlap=512)
tiles = tile_gen.generate_tiles_from_geotiff(
    clip_path,
    "temp/tiles/"
)

print(f"Generated {len(tiles)} tiles")

# ============================================
# STEP 6: YOLO Detection
# ============================================
detector = YOLODetector(
    model_path=model_path,
    conf_threshold=0.1,  # Initial threshold
    iou_threshold=0.45,
    image_size=tile_size
)

detections = detector.detect_tiles(
    tiles,
    batch_size=16,
    save_results=True,
    output_dir="temp/detections/"
)

total_dets = sum(d['num_detections'] for d in detections)
print(f"Total detections: {total_dets}")

# ============================================
# STEP 7: Merge Tile Detections
# ============================================
postprocessor = DetectionPostprocessor(iou_threshold=0.5)
merged = postprocessor.merge_tile_detections(
    detections,
    image_shape=(clip_height, clip_width)
)



2026-02-11 18:22:44,732 - INFO - Converting to grayscale 8-bit: datasets/D3C1218-401317F012_a_geo.tif
2026-02-11 18:22:52,375 - INFO - Input has 1 band(s)
2026-02-11 18:22:52,376 - INFO - Input is already grayscale, converting to 8-bit
2026-02-11 18:24:13,546 - INFO - Output array shape: (24063, 37240), dtype: uint8
2026-02-11 18:24:14,029 - INFO - Grayscale conversion completed: temp/image_gray.tif
2026-02-11 18:24:14,154 - INFO - Reprojecting to EPSG:4326: temp/image_gray.tif
2026-02-11 18:24:49,917 - INFO - Reprojection completed: temp/image_4326.tif
2026-02-11 18:24:49,918 - INFO - Processing to regular size with tile_size=1024, padding=False
2026-02-11 18:24:49,919 - INFO - Original size: 39356x20419
2026-02-11 18:24:49,919 - INFO - Clipped size: 38912x19456
2026-02-11 18:24:53,764 - INFO - Processed image saved to temp/image_clipped.tif
2026-02-11 18:24:53,804 - INFO - Initialized TileGenerator: tile_size=1024, overlap=512, stride=512
2026-02-11 18:24:53,804 - INFO - Generating g

Reference image: 38912×19456
Reference geotransform: (51.10509128836616, 1.2045833491161536e-05, 0.0, 35.41992762132754, 0.0, -1.2045833491161536e-05)


2026-02-11 18:24:59,185 - INFO - Generated 2775 georeferenced tiles
2026-02-11 18:24:59,218 - INFO - Saved tile metadata to temp/tiles/tile_metadata.json
2026-02-11 18:24:59,373 - INFO - Successfully loaded YOLO model from runs/detect/train29/weights/best.pt
2026-02-11 18:24:59,373 - INFO - Initialized YOLODetector on cuda
2026-02-11 18:24:59,374 - INFO - Model: runs/detect/train29/weights/best.pt
2026-02-11 18:24:59,374 - INFO - Conf threshold: 0.1, IOU threshold: 0.45
2026-02-11 18:24:59,374 - INFO - Image size: 1024
2026-02-11 18:24:59,375 - INFO - Running batch detection on 2775 images


Generated 2775 tiles


2026-02-11 18:25:44,896 - INFO - Completed batch detection: 2775 results
2026-02-11 18:25:44,980 - INFO - Merged 3235 detections to 1835 after NMS


Total detections: 3235


In [18]:
from qala_utils.utils import DetectionPostprocessor, QalaPipeline
from osgeo import gdal

# After merging detections
postprocessor = DetectionPostprocessor(iou_threshold=0.5, class_name="qala")
merged = postprocessor.merge_tile_detections(
    detections,
    image_shape=(clip_height, clip_width)
)

# Get reference geotransform
ref_ds = gdal.Open(clip_path)
reference_geotransform = ref_ds.GetGeoTransform()
ref_ds = None

# Save merged detections (BEFORE processing)
postprocessor.save_merged_detections(
    merged,
    reference_geotransform=reference_geotransform,
    output_path="results/merged_detections_before.gpkg",
    crs="EPSG:4326",
    driver="GPKG"
)

2026-02-11 18:11:22,817 - INFO - Merged 3235 detections to 1835 after NMS
2026-02-11 18:11:23,027 - INFO - Created 1,835 records
2026-02-11 18:11:23,031 - INFO - Saved 1835 merged detections (before processing) to results/merged_detections_before.gpkg


'results/merged_detections_before.gpkg'

In [21]:
from qala_utils.utils import DetectionPostprocessor, QalaPipeline
from osgeo import gdal
import numpy as np

# After detection step...
detections = detector.detect_tiles(tiles, batch_size=16)

# STEP 1: Merge tile detections
postprocessor = DetectionPostprocessor(
    iou_threshold=0.5,
    class_name="qala"
)
merged = postprocessor.merge_tile_detections(
    detections,
    image_shape=(clip_height, clip_width)
)

print(f"Merged: {merged['num_detections']} detections")

# STEP 2: Get geotransform
ref_ds = gdal.Open(clip_path)
reference_geotransform = ref_ds.GetGeoTransform()
ref_ds = None

# STEP 3: Process and filter
pipeline = QalaPipeline(
    min_confidence_qala=0.1,      # Filter low confidence detections
    overlap_threshold=0.9          # Merge if >90% overlap
)

gdf = pipeline.process_detections(
    merged,
    reference_geotransform=reference_geotransform,
    crs="EPSG:4326"
)

print(f"Final result: {len(gdf)} detections")

# STEP 4: Export
gdf.to_file("results/qala_detections.gpkg", driver="GPKG")

2026-02-11 18:56:33,576 - INFO - Running batch detection on 2775 images
2026-02-11 18:57:09,067 - INFO - Completed batch detection: 2775 results
2026-02-11 18:57:09,145 - INFO - Merged 3235 detections to 1835 after NMS
2026-02-11 18:57:09,148 - INFO - QalaPipeline initialized: min_confidence_qala=0.1, overlap_threshold=90.0%
2026-02-11 18:57:09,148 - INFO - ============================================================
2026-02-11 18:57:09,148 - INFO - QANAT DETECTION PROCESSING PIPELINE
2026-02-11 18:57:09,149 - INFO - ============================================================
2026-02-11 18:57:09,149 - INFO - Step 1: Converting detections to GeoDataFrame
2026-02-11 18:57:09,199 - INFO - Created GeoDataFrame with 1835 detections
2026-02-11 18:57:09,199 - INFO - Total detections: 1835
2026-02-11 18:57:09,200 - INFO - Classes: {'qala': 1835}
2026-02-11 18:57:09,201 - INFO - 
Step 2: Single-class pipeline - processing 'qala' detections
2026-02-11 18:57:09,203 - INFO - 
Step 3: Dissolving o

Merged: 1835 detections


2026-02-11 18:57:09,546 - INFO - Dissolved to 1646 bboxes (from 1835)
2026-02-11 18:57:09,547 - INFO - Merged 189 boxes with >90.0% overlap
2026-02-11 18:57:09,547 - INFO - 
Step 4: Filtering by confidence
2026-02-11 18:57:09,548 - INFO - Confidence filter: 1646 → 1646 (min=0.1)
2026-02-11 18:57:09,548 - INFO - ============================================================
2026-02-11 18:57:09,549 - INFO - FINAL RESULT: 1646 qala detections
2026-02-11 18:57:09,549 - INFO - ============================================================
2026-02-11 18:57:09,585 - INFO - Created 1,646 records


Final result: 1646 detections


In [ ]:
import zipfile
import os

folder_path = "/content/results"
zip_path = "/content/results.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, folder_path)
            zipf.write(full_path, arcname)


In [ ]:
# After processing detections
gdf = clusterer.process_detections(
    merged,
    reference_geotransform=reference_geotransform,
    crs="EPSG:4326"
)

# Export with centroids (includes bounding boxes + centroid columns)
clusterer.export_results(
    gdf,
    "results/qala_detections.gpkg",
    driver="GPKG",
    include_centroids=True  # Adds centroid_x, centroid_y, centroid_wkt columns
)

from ultralytics import YOLO

# Load a pretrained YOLO11n model
model = YOLO("/content/runs/detect/train2/weights/best.pt")

# Define path to the image file
source = "/content/datasets/images/test/"

# Run inference on the source
#model.predict(source, save=True, imgsz=256)
model.predict(source, save=True)


import os
from IPython.display import display, Image
directory = '/content/runs/detect/predict3'
for filename in os.listdir(directory):
    if filename.endswith(".jpg") or filename.endswith(".jpeg"):
        display(Image(filename=os.path.join(directory, filename)))

metrics = model.val(split='test', imgsz=512)


import os
import zipfile

def zip_folder_recursively(folder_path, output_zip_path):
    with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                # This ensures correct relative path inside the zip
                arcname = os.path.relpath(file_path, start=folder_path)
                zipf.write(file_path, arcname)
    print(f"Folder zipped recursively into: {output_zip_path}")

# Example usage:
folder_to_zip = '/content/runs/'  # Replace with your folder
zip_output = '/content/runs.zip'  # Output zip path

zip_folder_recursively(folder_to_zip, zip_output)